In [ ]:
# membersihkan sisa folder dari percobaan sebelumnya (aman jika tidak ada)
!rm -rf /content/_rels /content/xl /content/docProps "/content/[Content_Types].xml"
!pip -q install sentence-transformers openpyxl

In [ ]:
# ===== MESIN PENGECEKAN =====
import os
import re
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- 1. Preprocessing ----------
STOPWORDS = set("""
yang dan di ke dari untuk pada dengan dalam adalah ini itu atau juga serta oleh
sebagai dapat akan agar karena sehingga bahwa para tersebut terhadap antara
sebuah suatu ada tidak lebih sangat telah dalam secara hal saat masih
""".split())

def bersihkan(teks):
    teks = str(teks).lower()
    teks = re.sub(r"[^a-z0-9\s]", " ", teks)
    return " ".join(w for w in teks.split() if w not in STOPWORDS and len(w) > 1)

# ---------- 2. Ekstraksi aspek (rule-based dari kalimat) ----------
KATA_KUNCI = {
    "masalah": ["masalah", "permasalahan", "kesulitan", "tantangan", "kendala",
                "belum", "kurang", "sulit", "rendah", "tinggi", "meningkat", "menurun", "merugikan", "berhenti", "risiko",
                "terancam", "punah", "keterbatasan", "kurangnya", "hambatan", "fluktuasi", "menyulitkan"],
    "metode":  ["metode", "algoritma", "model", "pendekatan",
                "diterapkan", "klasifikasi", "regresi", "clustering", "forecasting",
                "random forest", "svm", "lstm", "naive bayes", "k-means", "xgboost",
                "arima", "cnn", "bert", "tf-idf", "cosine"],
    "data":    ["data", "dataset", "kuesioner", "responden", "sampel", "populasi",
                "sumber", "periode", "tahun", "record", "survei", "scraping", "twitter",
                "kaggle", "transaksi"],
}

def ekstrak_aspek(teks):
    kalimat = re.split(r"(?<=[.!?])\s+|\n", str(teks))
    hasil = {a: [] for a in KATA_KUNCI}
    for k in kalimat:
        kl = k.lower()
        for aspek, kws in KATA_KUNCI.items():
            if any(kw in kl for kw in kws):
                hasil[aspek].append(k)
    return {a: " ".join(v) for a, v in hasil.items()}

# ---------- 3. Kemiripan TF-IDF (cadangan jika SBERT tidak ada) ----------
def sim_per_kolom(korpus, query):
    korpus_bersih = [bersihkan(t) for t in korpus]
    q = bersihkan(query)
    if not q or not any(korpus_bersih):
        return [0.0] * len(korpus)
    vec = TfidfVectorizer(ngram_range=(1, 2))
    X = vec.fit_transform(korpus_bersih + [q])
    return cosine_similarity(X[-1], X[:-1]).ravel()

MODEL_DEFAULT = "paraphrase-multilingual-MiniLM-L12-v2"

# ---------- Kamus metode & sumber data ----------
KAMUS_METODE = {
    "random forest": r"random\s+forest", "xgboost": r"xgboost", "lightgbm": r"lightgbm",
    "naive bayes": r"na[iï]ve\s+bayes", "svm": r"svm|support\s+vector\s+machine",
    "k-means": r"k[\s\-]?means", "dbscan": r"dbscan", "lstm": r"lstm", "gru": r"gru",
    "cnn": r"cnn|convolutional", "bert": r"indo?bert|bert", "arima": r"arima|sarima",
    "prophet": r"prophet", "regresi linier": r"regresi\s+lin[ie]+r|linear\s+regression",
    "regresi logistik": r"regresi\s+logistik|logistic\s+regression",
    "decision tree": r"decision\s+tree|pohon\s+keputusan", "knn": r"knn|k[\s\-]?nearest",
    "apriori": r"apriori", "fp-growth": r"fp[\s\-]?growth", "yolo": r"yolo\w*",
    "efficientnet": r"efficientnet\w*", "mobilenet": r"mobilenet\w*", "resnet": r"resnet\w*",
    "transformer": r"transformer", "tf-idf": r"tf[\s\-]?idf", "word2vec": r"word2vec",
    "topsis": r"topsis", "ahp": r"ahp", "saw": r"saw", "rfm": r"rfm", "pca": r"pca",
    "smote": r"smote", "waterfall": r"waterfall", "scrum": r"scrum",
    "cosine similarity": r"cosine", "lda": r"lda|latent\s+dirichlet",
    "neural network": r"ann|neural\s+network|jaringan\s+saraf",
}
KAMUS_DATA = {
    "kaggle": r"kaggle", "twitter/x": r"twitter|tweet", "instagram": r"instagram",
    "tiktok": r"tiktok", "youtube": r"youtube", "google play": r"google\s+play|play\s+store",
    "scraping/crawling": r"scraping|crawling", "kuesioner": r"kuesioner|angket",
    "survei": r"survei|survey", "wawancara": r"wawancara", "citra satelit": r"citra\s+satelit|landsat|sentinel",
    "data saham": r"saham|harga\s+penutupan", "data transaksi": r"transaksi", "uci": r"uci",
    "bps": r"bps|badan\s+pusat\s+statistik", "rekam medis": r"rumah\s+sakit|rekam\s+medis",
    "citra/gambar": r"citra|gambar|image", "ulasan/komentar": r"ulasan|review|komentar",
}


def deteksi(teks, kamus):
    t = " " + re.sub(r"[^a-z0-9\s\-]", " ", str(teks).lower()) + " "
    return {nama for nama, pola in kamus.items()
            if re.search(r"(?<![a-z0-9])(?:" + pola + r")(?![a-z0-9])", t)}


# ---------- BM25 sederhana ----------
class BM25:
    def __init__(self, korpus_token, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(d) for d in korpus_token]
        self.dl = np.array([len(d) for d in korpus_token], dtype=float)
        self.avgdl = self.dl.mean() if len(self.dl) and self.dl.mean() > 0 else 1.0
        n = len(korpus_token)
        df = Counter(t for d in korpus_token for t in set(d))
        self.idf = {t: np.log((n - f + 0.5) / (f + 0.5) + 1) for t, f in df.items()}

    def skor_maks(self, query_token):
        """Skor acuan: query dianggap dokumen sepanjang rata-rata. Kata yang tidak ada di database
        dihitung dengan idf tertinggi, jadi query yang banyak kata barunya tidak bisa mencapai 1.0."""
        n = len(self.tf)
        idf_maks = np.log((n + 0.5) / 0.5 + 1)
        qtf = Counter(query_token)
        panjang = max(len(query_token), 1)
        total = 0.0
        for t, f in qtf.items():
            total += self.idf.get(t, idf_maks) * f * (self.k1 + 1) / (
                f + self.k1 * (1 - self.b + self.b * panjang / self.avgdl))
        return total

    def skor(self, query_token):
        s = np.zeros(len(self.tf))
        for t in set(query_token):
            if t not in self.idf:
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(t, 0)
                if f:
                    s[i] += self.idf[t] * f * (self.k1 + 1) / (
                        f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl))
        return s


def ekstrak_aspek_lanjut(teks):
    """Ekstraksi dasar (kata kunci) + kalimat yang memuat nama metode/sumber data dari kamus."""
    hasil = ekstrak_aspek(teks)
    for k in re.split(r"(?<=[.!?])\s+|\n", str(teks)):
        for aspek, kamus in (("metode", KAMUS_METODE), ("data", KAMUS_DATA)):
            if deteksi(k, kamus) and k not in hasil[aspek]:
                hasil[aspek] = (hasil[aspek] + " " + k).strip()
    return hasil


def _teks(v):
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v)


class AnalisisTopik:
    def __init__(self, df, alpha=0.7, pakai_sbert=True, model_name=MODEL_DEFAULT,
                 cache_dir=".cache_embedding", encoder=None):
        self.alpha, self.model_name, self.cache_dir = alpha, model_name, cache_dir
        self.mode, self._enc_fn = "tfidf", None
        if encoder is not None:
            self.mode, self._enc_fn = "sbert", encoder
        elif pakai_sbert:
            try:
                from sentence_transformers import SentenceTransformer
                print(f"[1/3] Memuat model {model_name} (pertama kali bisa lama: mengunduh model)...")
                m = SentenceTransformer(model_name)
                self._enc_fn = lambda t: m.encode(t, normalize_embeddings=True,
                                                  show_progress_bar=False, batch_size=32)
                self.mode = "sbert"
            except Exception as e:
                print(f"[PERINGATAN] SBERT tidak tersedia ({type(e).__name__}). Memakai TF-IDF sebagai pengganti.")
        # ambang: skala skor SBERT berbeda dengan TF-IDF
        self.ambang_mirip, self.ambang_bahaya, self.ambang_aspek = (
            (0.60, 0.75, 0.55) if self.mode == "sbert" else (0.30, 0.50, 0.25))
        self._siapkan(df)

    # ---------- persiapan database ----------
    def _siapkan(self, df):
        d = df.copy().reset_index(drop=True)
        for k in ("judul", "latar_belakang", "abstrak"):
            if k not in d.columns:
                d[k] = ""
            d[k] = d[k].apply(_teks)
        d["latar_belakang"] = d["latar_belakang"].str[:6000]          # Chapter1 bisa sangat panjang
        asp_abs = (d["judul"] + ". " + d["abstrak"]).apply(ekstrak_aspek_lanjut)
        asp_lb = d["latar_belakang"].apply(ekstrak_aspek_lanjut)
        for a in KATA_KUNCI:
            eks = d[a].apply(_teks) if a in d.columns else pd.Series([""] * len(d))
            urut = (asp_lb, asp_abs) if a == "masalah" else (asp_abs, asp_lb)   # masalah: latar belakang; metode/data: abstrak
            otomatis = [x[a] or y[a] for x, y in zip(*urut)]
            d[a] = [(e if e.strip() else o)[:700] for e, o in zip(eks, otomatis)]
        d["teks_semantik"] = d["judul"] + ". " + d["abstrak"]
        d["teks_leksikal"] = d["judul"] + " " + d["abstrak"] + " " + d["latar_belakang"]
        if "tahun" in d.columns:
            def _th(v):
                try:
                    return str(int(float(v)))
                except (TypeError, ValueError):
                    return _teks(v)
            d["tahun"] = d["tahun"].apply(_th)
        d["jurusan"] = d["jurusan"].apply(_teks) if "jurusan" in d.columns else "-"
        d["sumber_file"] = d["sumber_file"].apply(_teks) if "sumber_file" in d.columns else "-"
        self.df, self.n = d, len(d)

        tok = [bersihkan(t).split() for t in d["teks_leksikal"]]
        self.bm25 = BM25(tok)
        self.df_token = Counter(t for doc in tok for t in set(doc))
        self.metode_doc = [deteksi(t, KAMUS_METODE) for t in d["teks_leksikal"]]
        self.data_doc = [deteksi(t, KAMUS_DATA) for t in d["teks_leksikal"]]

        if self.mode == "sbert":
            print(f"[2/3] Menghitung embedding {self.n} topik (disimpan di cache)...")
            self.emb_sem = self._encode(d["teks_semantik"].tolist(), "sem")
            self.emb_asp = {a: self._encode(d[a].tolist(), a) for a in KATA_KUNCI}
            self.ada_asp = {a: np.array([bool(x.strip()) for x in d[a]]) for a in KATA_KUNCI}
        print(f"[3/3] Siap. Mode: {self.mode.upper()} | {self.n} topik | alpha={self.alpha}\n")

    def _encode(self, teks, nama):
        kunci = hashlib.md5((self.model_name + "||".join(teks)).encode("utf-8")).hexdigest()
        path = os.path.join(self.cache_dir, f"{nama}_{kunci}.npy")
        if os.path.exists(path):
            return np.load(path)
        emb = np.asarray(self._enc_fn([t if t.strip() else " " for t in teks]), dtype=float)
        os.makedirs(self.cache_dir, exist_ok=True)
        np.save(path, emb)
        return emb

    # ---------- pencarian ----------
    def cari(self, judul, deskripsi="", top_k=5, masalah="", metode="", data=""):
        """Boleh isi `deskripsi` bebas, atau isian terpisah `masalah`, `metode`, `data` (lebih akurat)."""
        q = ". ".join(b.strip() for b in (judul, masalah, metode, data, deskripsi) if b and b.strip())
        q_asp = ekstrak_aspek_lanjut(q)
        for nama, isi in (("masalah", masalah), ("metode", metode), ("data", data)):
            if isi and isi.strip():
                q_asp[nama] = isi.strip()
        nol = np.zeros(self.n)

        if self.mode == "sbert":
            qe = np.asarray(self._enc_fn([q]), dtype=float)[0]
            sem = np.clip(self.emb_sem @ qe, 0, 1)
            asp = {}
            for a in KATA_KUNCI:
                if q_asp[a].strip():
                    ae = np.asarray(self._enc_fn([q_asp[a]]), dtype=float)[0]
                    asp[a] = np.where(self.ada_asp[a], np.clip(self.emb_asp[a] @ ae, 0, 1), 0.0)
                else:
                    asp[a] = nol
        else:
            sem = np.array(sim_per_kolom(self.df["teks_semantik"], q))
            asp = {a: (np.array(sim_per_kolom(self.df[a], q_asp[a])) if q_asp[a] else nol)
                   for a in KATA_KUNCI}

        q_token = bersihkan(q).split()
        bm = self.bm25.skor(q_token)
        acuan = self.bm25.skor_maks(q_token)
        bm_norm = np.clip(bm / acuan, 0, 1) if acuan > 0 else nol
        hybrid = self.alpha * sem + (1 - self.alpha) * bm_norm

        idx = np.argsort(hybrid)[::-1][:top_k]
        q_metode = deteksi(f"{judul}. {metode}" if metode.strip() else q, KAMUS_METODE)
        q_data = deteksi(data if data.strip() else q, KAMUS_DATA)
        # frasa bebas dari isian metode/dataset (untuk istilah yang tidak ada di kamus)
        frasa = {}
        gudang = self.df["teks_leksikal"].str.lower()
        for p in re.split(r",|;|/|\bdan\b|\bserta\b", f"{metode},{data}"):
            p = p.strip().lower()
            if len(p) >= 3:
                frasa[p] = int(gudang.str.contains(re.escape(p)).sum())
        q_tok = set(bersihkan(q).split())

        baris = []
        for i in idx:
            r = self.df.iloc[i]
            skor = float(hybrid[i])
            tingkat = "BAHAYA" if skor >= self.ambang_bahaya else "MIRIP" if skor >= self.ambang_mirip else "RENDAH"
            asp_tinggi = {a: float(asp[a][i]) >= self.ambang_aspek for a in KATA_KUNCI}
            asp_tinggi["metode"] = asp_tinggi["metode"] or bool(q_metode & self.metode_doc[i])   # kamus juga dihitung
            asp_tinggi["data"] = asp_tinggi["data"] or bool(q_data & self.data_doc[i])
            # kata kunci yang sama (utamakan yang jarang muncul di database)
            bersama = sorted(q_tok & set(bersihkan(r["teks_leksikal"]).split()),
                             key=lambda t: self.df_token[t])
            bersama = [t for t in bersama if self.df_token[t] / self.n < 0.5][:8]
            baris.append({
                "judul_existing": r["judul"], "sumber_file": r["sumber_file"], "jurusan": r["jurusan"],
                "tahun": r["tahun"] if "tahun" in self.df.columns else "",
                "skor_hybrid": round(skor, 3), "skor_semantik": round(float(sem[i]), 3),
                "skor_bm25": round(float(bm_norm[i]), 3),
                "masalah": round(float(asp["masalah"][i]), 3), "metode": round(float(asp["metode"][i]), 3),
                "data": round(float(asp["data"][i]), 3), "tingkat": tingkat,
                "metode_sama": ", ".join(sorted(q_metode & self.metode_doc[i])) or "-",
                "data_sama": ", ".join(sorted(q_data & self.data_doc[i])) or "-",
                "kata_kunci_sama": ", ".join(bersama) or "-",
                "saran": self._saran(asp_tinggi, tingkat),
            })
        hasil = pd.DataFrame(baris)
        return {"hasil": hasil, "aspek_input": q_asp, "metode_input": sorted(q_metode),
                "data_input": sorted(q_data), "skor_semua": hybrid, "frasa": frasa, **self._ringkasan(hasil, q_metode, q_data, hybrid)}

    # ---------- analisis tambahan ----------
    @staticmethod
    def _saran(t, tingkat):
        m, me, d = t["masalah"], t["metode"], t["data"]
        if m and me and d:
            return "Masalah, metode, dan data sama: berisiko dianggap duplikasi. Ubah salah satu secara nyata."
        if m and me:
            return "Masalah & metode sama: ganti dataset/objek studi, atau bandingkan dengan metode lain."
        if me and d:
            return "Metode & data sama: angkat rumusan masalah atau objek yang berbeda."
        if m and d:
            return "Masalah & data sama: gunakan metode/algoritma yang berbeda."
        if m:
            return "Hanya masalah sama: wajar, pastikan metode dan data berbeda."
        if me:
            return "Hanya metode sama: wajar, metode umum dipakai banyak topik."
        if d:
            return "Hanya data sama: pastikan masalah dan metode berbeda."
        return "Tidak ada aspek yang menonjol." if tingkat == "RENDAH" else "Mirip secara keseluruhan, tetapi tidak ada aspek tunggal yang dominan."

    def _ringkasan(self, hasil, q_metode, q_data, hybrid):
        n_mirip = int((hybrid >= self.ambang_mirip).sum())
        n_bahaya = int((hybrid >= self.ambang_bahaya).sum())
        if n_bahaya:
            verdict = "PERINGATAN DUPLIKASI: ada topik yang sangat mirip. Ubah arah topik."
        elif n_mirip == 0:
            verdict = "RESEARCH GAP: tidak ada topik di database yang melewati ambang kemiripan. Ide tampak orisinal."
        elif n_mirip < 3:
            verdict = f"IDE PASARAN RINGAN: {n_mirip} topik mirip. Perjelas kebaruan di metode atau data."
        else:
            verdict = f"RED OCEAN: {n_mirip} topik mirip. Area ini sudah sering diteliti."

        mirip_idx = np.where(hybrid >= self.ambang_mirip)[0]
        jurusan = Counter(self.df.loc[mirip_idx, "jurusan"]).most_common(3)
        tahun = Counter(self.df.loc[mirip_idx, "tahun"].astype(str)) if "tahun" in self.df.columns else Counter()

        freq_metode = {m: sum(m in s for s in self.metode_doc) for m in q_metode}
        freq_data = {x: sum(x in s for s in self.data_doc) for x in q_data}
        kombinasi = []
        if q_metode and q_data:
            for i in range(self.n):
                if q_metode & self.metode_doc[i] and q_data & self.data_doc[i]:
                    kombinasi.append(self.df.loc[i, "judul"])
        return {"verdict": verdict, "n_mirip": n_mirip, "n_bahaya": n_bahaya, "jurusan": jurusan,
                "tahun": tahun, "freq_metode": freq_metode, "freq_data": freq_data, "kombinasi": kombinasi}


# ---------- Laporan ----------
def buat_laporan(res, judul, deskripsi, n_db):
    L = ["=" * 70, "LAPORAN ANALISIS KEMIRIPAN TOPIK", "=" * 70,
         f"Judul     : {judul}", f"Deskripsi : {deskripsi}", f"Database  : {n_db} topik", ""]
    L.append(">> ASPEK TERDETEKSI DARI INPUT")
    for k, v in res["aspek_input"].items():
        L.append(f"   {k:8}: {v or '(tidak terdeteksi)'}")
    L.append(f"   metode   : {', '.join(res['metode_input']) or '(tidak ada di kamus)'}")
    L.append(f"   sumber data: {', '.join(res['data_input']) or '(tidak ada di kamus)'}")
    L += ["", ">> TOP HASIL"]
    for i, r in res["hasil"].iterrows():
        L += [f"{i+1}. [{r['tingkat']}] hybrid={r['skor_hybrid']}  (semantik={r['skor_semantik']}, bm25={r['skor_bm25']})",
              f"   Judul    : {r['judul_existing']}",
              f"   Asal     : {r['sumber_file']} | {r['jurusan']} | {r['tahun']}",
              f"   Aspek    : masalah={r['masalah']}  metode={r['metode']}  data={r['data']}",
              f"   Metode sama: {r['metode_sama']} | Data sama: {r['data_sama']}",
              f"   Kata kunci sama: {r['kata_kunci_sama']}",
              f"   Saran    : {r['saran']}", ""]
    L.append(">> PEMAKAIAN METODE & DATA DI DATABASE")
    for m, n in res["freq_metode"].items():
        L.append(f"   metode '{m}' dipakai di {n} dari {n_db} topik")
    for x, n in res["freq_data"].items():
        L.append(f"   data '{x}' muncul di {n} dari {n_db} topik")
    if res["kombinasi"]:
        L.append(f"   KOMBINASI metode+data yang sama ditemukan di {len(res['kombinasi'])} topik:")
        L += [f"     - {j}" for j in res["kombinasi"][:5]]
    elif res["metode_input"] and res["data_input"]:
        L.append("   Kombinasi metode+data ini belum ada di database (nilai tambah).")
    for f_, n in res.get("frasa", {}).items():
        L.append(f"   frasa '{f_}' muncul di {n} dari {n_db} topik")
    L += ["", ">> TREN"]
    L.append("   Jurusan paling banyak mirip: " + (", ".join(f"{j} ({n})" for j, n in res["jurusan"]) or "-"))
    if res["tahun"]:
        L.append("   Tahun topik mirip: " + ", ".join(f"{t} ({n})" for t, n in sorted(res["tahun"].items())))
    L += ["", ">> KESIMPULAN", "   " + res["verdict"], "=" * 70]
    return "\n".join(L)

In [ ]:
# ===== PATCH: bandingkan berdasarkan Judul + Abstrak + RUANG LINGKUP (Latar Belakang Bab 1 TIDAK dipakai) =====
BOBOT_SEM = {"abstrak": 0.5, "lingkup": 0.5}     # bobot kemiripan makna: abstrak vs ruang lingkup

# Heading Bab 1 bernomor: "1.4 Ruang Lingkup", "1.4. Ruang Lingkup", "1.3. Ruang Lingkup Skripsi", dst.
_POLA_NOMOR = re.compile(r"(?<![\d.])1\.\d{1,2}\.?\s+(?=[A-Za-z])")
_JUDUL_BAGIAN = {
    "rumusan": r"(?:rumusan|perumusan|identifikasi)\s+masalah",
    "lingkup": r"ruang\s+lingkup|batasan(?:\s+masalah|\s+penelitian)?",
}


def pecah_bab1(teks, maks=1500):
    """Ambil bagian 'Rumusan Masalah' dan 'Ruang Lingkup' dari Bab 1. Latar Belakang dibuang.
    Setiap heading bernomor (1.x) dianggap batas bagian, jadi bagian berhenti di heading berikutnya."""
    teks = re.sub(r"\s+", " ", _teks(teks))
    teks = re.sub(r"\d*\s*Universitas Kristen Petra\s*-*", " ", teks)       # sisa footer halaman
    mulai = [m for m in _POLA_NOMOR.finditer(teks)
             if teks[m.end():m.end() + 1].isupper()]                # heading selalu berhuruf kapital
    hasil = {k: "" for k in _JUDUL_BAGIAN}
    for i, m in enumerate(mulai):
        judul_bagian = teks[m.end():m.end() + 60]
        for k, pola in _JUDUL_BAGIAN.items():
            h = re.match(pola, judul_bagian, re.I)
            if h:
                akhir = mulai[i + 1].start() if i + 1 < len(mulai) else len(teks)
                hasil[k] = (hasil[k] + " " + teks[m.end() + h.end():akhir].strip()).strip()
    return {k: v[:maks] for k, v in hasil.items()}


def _siapkan_baru(self, df):
    d = df.copy().reset_index(drop=True)
    for k in ("judul", "latar_belakang", "abstrak"):
        if k not in d.columns:
            d[k] = ""
        d[k] = d[k].apply(_teks)
    bagian = d["latar_belakang"].apply(pecah_bab1)          # kolom ini berisi Bab 1 (Chapter1) utuh
    d["rumusan"] = [b["rumusan"] for b in bagian]
    d["lingkup"] = [b["lingkup"] for b in bagian]

    asp_abs = (d["judul"] + ". " + d["abstrak"]).apply(ekstrak_aspek_lanjut)
    asp_lin = d["lingkup"].apply(ekstrak_aspek_lanjut)
    for a in KATA_KUNCI:
        eks = d[a].apply(_teks) if a in d.columns else pd.Series([""] * len(d))
        if a == "masalah":      # masalah: rumusan masalah (bersih), cadangan abstrak
            otomatis = [r or x[a] for r, x in zip(d["rumusan"], asp_abs)]
        else:                   # metode & data: ruang lingkup + abstrak
            otomatis = [((lk[a] + " " + x[a]).strip() or li[:700])
                        for lk, x, li in zip(asp_lin, asp_abs, d["lingkup"])]
        d[a] = [(e if e.strip() else o)[:700] for e, o in zip(eks, otomatis)]

    d["teks_semantik"] = d["judul"] + ". " + d["abstrak"]
    d["teks_leksikal"] = d["judul"] + " " + d["abstrak"] + " " + d["lingkup"] + " " + d["rumusan"]
    if "tahun" in d.columns:
        def _th(v):
            try:
                return str(int(float(v)))
            except (TypeError, ValueError):
                return _teks(v)
        d["tahun"] = d["tahun"].apply(_th)
    d["jurusan"] = d["jurusan"].apply(_teks) if "jurusan" in d.columns else "-"
    d["sumber_file"] = d["sumber_file"].apply(_teks) if "sumber_file" in d.columns else "-"
    self.df, self.n = d, len(d)
    self.ada_ling = np.array([bool(x.strip()) for x in d["lingkup"]])

    n_ling, n_rum = int(self.ada_ling.sum()), int(sum(bool(x) for x in d["rumusan"]))
    print(f"Bab 1 diproses: Ruang Lingkup terbaca {n_ling}/{self.n} | Rumusan Masalah terbaca {n_rum}/{self.n}")
    if n_ling < self.n:
        print(f"[CATATAN] {self.n - n_ling} topik tidak punya Ruang Lingkup terbaca -> dibandingkan lewat abstrak saja "
              "(Latar Belakang tidak dipakai).")

    tok = [bersihkan(t).split() for t in d["teks_leksikal"]]
    self.bm25 = BM25(tok)
    self.df_token = Counter(t for doc in tok for t in set(doc))
    self.metode_doc = [deteksi(t, KAMUS_METODE) for t in d["teks_leksikal"]]
    self.data_doc = [deteksi(t, KAMUS_DATA) for t in d["teks_leksikal"]]

    if self.mode == "sbert":
        print(f"Menghitung embedding {self.n} topik (disimpan di cache)...")
        self.emb_sem = self._encode(d["teks_semantik"].tolist(), "sem")
        self.emb_ling = self._encode(d["lingkup"].tolist(), "ling")
        self.emb_asp = {a: self._encode(d[a].tolist(), a) for a in KATA_KUNCI}
        self.ada_asp = {a: np.array([bool(x.strip()) for x in d[a]]) for a in KATA_KUNCI}
    print(f"Siap. Mode: {self.mode.upper()} | {self.n} topik | alpha={self.alpha}\n")


def cari_baru(self, judul, deskripsi="", top_k=5, masalah="", metode="", data=""):
    q = ". ".join(b.strip() for b in (judul, masalah, metode, data, deskripsi) if b and b.strip())
    # untuk ruang lingkup: bandingkan metode + dataset (bukan latar belakang/masalah)
    q_ling = ". ".join(b.strip() for b in (judul, metode, data) if b and b.strip()) if (metode.strip() or data.strip()) else q
    q_asp = ekstrak_aspek_lanjut(q)
    for nama, isi in (("masalah", masalah), ("metode", metode), ("data", data)):
        if isi and isi.strip():
            q_asp[nama] = isi.strip()
    nol = np.zeros(self.n)
    wa, wl = BOBOT_SEM["abstrak"], BOBOT_SEM["lingkup"]

    if self.mode == "sbert":
        qe = np.asarray(self._enc_fn([q]), dtype=float)[0]
        sem_abs = np.clip(self.emb_sem @ qe, 0, 1)
        ql = np.asarray(self._enc_fn([q_ling]), dtype=float)[0]
        sem_ling = np.clip(self.emb_ling @ ql, 0, 1)
        asp = {}
        for a in KATA_KUNCI:
            if q_asp[a].strip():
                ae = np.asarray(self._enc_fn([q_asp[a]]), dtype=float)[0]
                asp[a] = np.where(self.ada_asp[a], np.clip(self.emb_asp[a] @ ae, 0, 1), 0.0)
            else:
                asp[a] = nol
    else:
        sem_abs = np.array(sim_per_kolom(self.df["teks_semantik"], q))
        sem_ling = np.array(sim_per_kolom(self.df["lingkup"], q_ling))
        asp = {a: (np.array(sim_per_kolom(self.df[a], q_asp[a])) if q_asp[a] else nol) for a in KATA_KUNCI}
    # topik tanpa Ruang Lingkup terbaca: hanya pakai abstrak
    sem = np.where(self.ada_ling, wa * sem_abs + wl * sem_ling, sem_abs)

    q_token = bersihkan(q).split()
    bm = self.bm25.skor(q_token)
    acuan = self.bm25.skor_maks(q_token)
    bm_norm = np.clip(bm / acuan, 0, 1) if acuan > 0 else nol
    hybrid = self.alpha * sem + (1 - self.alpha) * bm_norm

    idx = np.argsort(hybrid)[::-1][:top_k]
    q_metode = deteksi(f"{judul}. {metode}" if metode.strip() else q, KAMUS_METODE)
    q_data = deteksi(data if data.strip() else q, KAMUS_DATA)
    frasa = {}
    gudang = self.df["teks_leksikal"].str.lower()
    for p in re.split(r",|;|/|\bdan\b|\bserta\b", f"{metode},{data}"):
        p = p.strip().lower()
        if len(p) >= 3:
            frasa[p] = int(gudang.str.contains(re.escape(p)).sum())
    q_tok = set(bersihkan(q).split())

    baris = []
    for i in idx:
        r = self.df.iloc[i]
        skor = float(hybrid[i])
        tingkat = "BAHAYA" if skor >= self.ambang_bahaya else "MIRIP" if skor >= self.ambang_mirip else "RENDAH"
        asp_tinggi = {a: float(asp[a][i]) >= self.ambang_aspek for a in KATA_KUNCI}
        asp_tinggi["metode"] = asp_tinggi["metode"] or bool(q_metode & self.metode_doc[i])
        asp_tinggi["data"] = asp_tinggi["data"] or bool(q_data & self.data_doc[i])
        bersama = sorted(q_tok & set(bersihkan(r["teks_leksikal"]).split()), key=lambda t: self.df_token[t])
        bersama = [t for t in bersama if self.df_token[t] / self.n < 0.5][:8]
        baris.append({
            "judul_existing": r["judul"], "sumber_file": r["sumber_file"], "jurusan": r["jurusan"],
            "tahun": r["tahun"] if "tahun" in self.df.columns else "",
            "skor_hybrid": round(skor, 3), "skor_semantik": round(float(sem[i]), 3),
            "skor_abstrak": round(float(sem_abs[i]), 3),
            "skor_lingkup": round(float(sem_ling[i]), 3) if self.ada_ling[i] else np.nan,
            "skor_bm25": round(float(bm_norm[i]), 3),
            "masalah": round(float(asp["masalah"][i]), 3), "metode": round(float(asp["metode"][i]), 3),
            "data": round(float(asp["data"][i]), 3), "tingkat": tingkat,
            "metode_sama": ", ".join(sorted(q_metode & self.metode_doc[i])) or "-",
            "data_sama": ", ".join(sorted(q_data & self.data_doc[i])) or "-",
            "kata_kunci_sama": ", ".join(bersama) or "-",
            "saran": self._saran(asp_tinggi, tingkat),
        })
    hasil = pd.DataFrame(baris)
    return {"hasil": hasil, "aspek_input": q_asp, "metode_input": sorted(q_metode),
            "data_input": sorted(q_data), "skor_semua": hybrid, "frasa": frasa,
            **self._ringkasan(hasil, q_metode, q_data, hybrid)}


AnalisisTopik._siapkan = _siapkan_baru
AnalisisTopik.cari = cari_baru
print("Patch aktif: pembanding = Judul + Abstrak + Ruang Lingkup. Latar Belakang Bab 1 tidak dipakai.")

Patch aktif: pembanding = Judul + Abstrak + Ruang Lingkup. Latar Belakang Bab 1 tidak dipakai.


In [ ]:
import glob, os
import pandas as pd

DATA_DIR = "/content"     # semua file .xlsx/.csv di folder ini (termasuk subfolder) ikut dibaca

ALIAS = {"chapter1": "latar_belakang", "chapter_1": "latar_belakang", "bab1": "latar_belakang",
         "bab_1": "latar_belakang", "abstract": "abstrak", "title": "judul", "year": "tahun",
         "department": "jurusan", "prodi": "jurusan"}
FOLDER_UMUM = {"content", "data", "sample_data"}

def muat_data(folder):
    frames, dibaca = [], []
    for path in sorted(glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)):
        if not path.lower().endswith((".xlsx", ".xls", ".csv")):
            continue
        try:
            bagian = {"": pd.read_csv(path)} if path.lower().endswith(".csv") else pd.read_excel(path, sheet_name=None)
        except Exception as e:
            print(f"[GAGAL] {os.path.basename(path)}: {e}"); continue
        for sheet, df in bagian.items():
            kolom = [ALIAS.get(c, c) for c in (str(c).strip().lower().replace(" ", "_") for c in df.columns)]
            if "judul" not in kolom:
                continue                       # bukan data skripsi (mis. file contoh bawaan Colab)
            df = df.copy(); df.columns = kolom
            df = df.loc[:, ~pd.Index(df.columns).duplicated()]
            for k in ("latar_belakang", "abstrak"):
                if k not in df.columns: df[k] = ""
            df["judul"] = df["judul"].astype(str).str.strip()
            df = df[df["judul"].str.len() > 3]
            induk = os.path.basename(os.path.dirname(os.path.abspath(path)))
            stem = os.path.splitext(os.path.basename(path))[0]
            if "jurusan" not in df.columns:
                df["jurusan"] = stem if induk.lower() in FOLDER_UMUM else induk
            df["sumber_file"] = os.path.basename(path)
            frames.append(df); dibaca.append((os.path.basename(path), sheet or "-", len(df)))
    if not frames:
        raise FileNotFoundError("Tidak ada file skripsi (.xlsx/.csv dengan kolom 'Judul') di " + folder +
                                ". Seret file ke panel Files di kiri, lalu jalankan ulang sel ini.")
    db = pd.concat(frames, ignore_index=True).drop_duplicates(subset="judul").reset_index(drop=True)
    print("File terbaca:")
    for f, s, n in dibaca:
        print(f"  - {f} (sheet: {s}) -> {n} topik")
    return db

db = muat_data(DATA_DIR)
print(f"\nTotal database: {len(db)} topik")
display(db.groupby("jurusan").agg(jumlah_topik=("judul", "count")))
db[[c for c in ("judul", "jurusan", "tahun") if c in db.columns]].head(5)

File terbaca:
  - DSA.xlsx (sheet: Skripsi) -> 54 topik
  - INFOR.xlsx (sheet: Skripsi_Info) -> 612 topik
  - SIB.xlsx (sheet: Skripsi_Batch2) -> 226 topik

Total database: 778 topik


,jumlah_topik
jurusan,
DSA,54
INFOR,518
SIB,206


,judul,jurusan,tahun
0,Aplikasi pembelajaran dan pengenalan aksara Su...,DSA,2025.0
1,Implemantasi data analytics terhadap data lowo...,DSA,2025.0
2,Optimasi algoritma search engine pada website ...,DSA,2025.0
3,Demand forecasting dengan SARIMAX untuk stock ...,DSA,2025.0
4,Aspect based sentiment analysis terhadap produ...,DSA,2025.0


In [ ]:
mesin = AnalisisTopik(db, alpha=0.7)   # alpha = bobot makna (SBERT); sisanya BM25
# Ambang bisa diubah setelah dicoba pada data Anda:
# mesin.ambang_mirip, mesin.ambang_bahaya = 0.60, 0.75

[1/3] Memuat model paraphrase-multilingual-MiniLM-L12-v2 (pertama kali bisa lama: mengunduh model)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Bab 1 diproses: Ruang Lingkup terbaca 712/778 | Rumusan Masalah terbaca 701/778
[CATATAN] 66 topik tidak punya Ruang Lingkup terbaca -> dibandingkan lewat abstrak saja (Latar Belakang tidak dipakai).
Menghitung embedding 778 topik (disimpan di cache)...
Siap. Mode: SBERT | 778 topik | alpha=0.7



In [ ]:
#@title Masukkan ide topik { display-mode: "form" }
judul = "Pengenalan Aksara Sunda Menggunakan YOLOv8"  #@param {type:"string"}
masalah = "Aksara Sunda terancam punah karena kurangnya media belajar yang interaktif"  #@param {type:"string"}
metode = "YOLOv8 dan EfficientNetV2B0"  #@param {type:"string"}
dataset = "Dataset gambar aksara Sunda"  #@param {type:"string"}
jumlah_hasil = 5  #@param {type:"slider", min:1, max:10, step:1}

res = mesin.cari(judul, "", top_k=jumlah_hasil, masalah=masalah, metode=metode, data=dataset)
ringkas = f"Masalah: {masalah} | Metode: {metode} | Dataset: {dataset}"
print(buat_laporan(res, judul, ringkas, mesin.n))
res["hasil"][["judul_existing", "jurusan", "tahun", "skor_hybrid", "tingkat", "skor_abstrak", "skor_lingkup", "masalah", "metode", "data"]]

LAPORAN ANALISIS KEMIRIPAN TOPIK
Judul     : Pengenalan Aksara Sunda Menggunakan YOLOv8
Deskripsi : Masalah: Aksara Sunda terancam punah karena kurangnya media belajar yang interaktif | Metode: YOLOv8 dan EfficientNetV2B0 | Dataset: Dataset gambar aksara Sunda
Database  : 778 topik

>> ASPEK TERDETEKSI DARI INPUT
   masalah : Aksara Sunda terancam punah karena kurangnya media belajar yang interaktif
   metode  : YOLOv8 dan EfficientNetV2B0
   data    : Dataset gambar aksara Sunda
   metode   : efficientnet, yolo
   sumber data: citra/gambar

>> TOP HASIL
1. [RENDAH] hybrid=0.572  (semantik=0.554, bm25=0.614)
   Judul    : Aplikasi pembelajaran dan pengenalan aksara Sunda menggunakan Arsitektur EfficientNetV2B0
   Asal     : DSA.xlsx | DSA | 2025
   Aspek    : masalah=0.326  metode=0.614  data=0.493
   Metode sama: efficientnet, yolo | Data sama: citra/gambar
   Kata kunci sama: efficientnetv2b0, sunda, aksara, yolov8, pengenalan, interaktif, kurangnya, media
   Saran    : Metode & data

,judul_existing,jurusan,tahun,skor_hybrid,tingkat,skor_abstrak,skor_lingkup,masalah,metode,data
0,Aplikasi pembelajaran dan pengenalan aksara Su...,DSA,2025,0.572,RENDAH,0.686,0.423,0.326,0.614,0.493
1,Implementasi Yolov8 untuk Character recognitio...,DSA,2025,0.357,RENDAH,0.383,0.412,0.113,0.552,0.434
2,Aplikasi pembelajaran perakitan komputer denga...,INFOR,2025,0.339,RENDAH,0.444,NaN,0.513,0.000,0.122
3,Aplikasi berbasis web untuk mengenali tipe dan...,INFOR,2023,0.316,RENDAH,0.389,0.417,0.247,0.468,0.461
4,Penerapan YOLOV8 dalam pembuatan aplikasi pemb...,INFOR,2025,0.307,RENDAH,0.438,0.235,0.310,0.260,0.392


In [ ]:
# Format tiap baris: (judul, masalah, metode, dataset)
daftar = [
    ("Prediksi churn nasabah bank dengan XGBoost", "Nasabah berhenti memakai layanan bank", "XGBoost", "Dataset nasabah dari Kaggle"),
    ("Analisis kepuasan layanan perpustakaan", "Kepuasan mahasiswa belum diketahui", "Statistik deskriptif", "Kuesioner 200 responden"),
]
baris = []
for j, m, me, d in daftar:
    r = mesin.cari(j, "", top_k=1, masalah=m, metode=me, data=d)
    t = r["hasil"].iloc[0]
    baris.append({"judul_input": j, "top_match": t["judul_existing"], "jurusan": t["jurusan"],
                  "skor_hybrid": t["skor_hybrid"], "tingkat": t["tingkat"], "kesimpulan": r["verdict"]})
hasil_batch = pd.DataFrame(baris)
hasil_batch.to_csv("/content/hasil_batch.csv", index=False)
hasil_batch

,judul_input,top_match,jurusan,skor_hybrid,tingkat,kesimpulan
0,Prediksi churn nasabah bank dengan XGBoost,Prediksi kelayakan pinjaman bank menggunakan m...,INFOR,0.484,RENDAH,RESEARCH GAP: tidak ada topik di database yang...
1,Analisis kepuasan layanan perpustakaan,Leads categorization dengan support vector mac...,INFOR,0.390,RENDAH,RESEARCH GAP: tidak ada topik di database yang...
